# 07 — Plasma denting and CHF beam profiles

Reproduces Fig. 3 of Timmis 2026: plasma surface denting reshapes the
reflected harmonic beam from a Gaussian-filtered profile (low intensity)
to a modulated, curved beam at high intensity.  The dent depth scales
non-linearly with a₀ via the Vincenti/Timmis formulas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from harmonyemissions import Laser, Target, simulate
from harmonyemissions.config import NumericsConfig
from harmonyemissions.viz import plot_beam_profile, plot_dent_map

In [ ]:
results = {}
numerics = NumericsConfig(pipeline_grid=96, pipeline_dx_um=0.1,
                          diag_harmonics=(1, 30))
for a0 in [3.0, 10.0, 24.0, 50.0]:
    r = simulate(Laser(a0=a0, spatial_profile='super_gaussian', spot_fwhm_um=2.0,
                       super_gaussian_order=8, angle_deg=45.0),
                 Target.sio2(t_HDR_fs=351.0), model='surface_pipeline', numerics=numerics)
    results[a0] = r
    print(f'a0={a0:5.1f}  dent_peak={r.diagnostics["dent_peak_lambda"]:.3f} λ  L={r.diagnostics["L_over_lambda"]:.3f} λ')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (a0, r) in zip(axes, results.items()):
    plot_dent_map(r, ax=ax)
    ax.set_title(f'a0 = {a0}')
fig.tight_layout()
plt.show()

In [ ]:
# Dent peak vs a₀ — compare to paper Fig. 3c (denting grows rapidly).
a0s = np.array(list(results.keys()))
peaks = np.array([results[a].diagnostics['dent_peak_lambda'] for a in a0s])
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(a0s, peaks, 'o-')
ax.set_xlabel('a₀')
ax.set_ylabel('peak dent / λ')
ax.set_title('Plasma dent depth vs driving intensity')
plt.show()